# Финальный pipeline — часть 2: расчёт метрик

Запускать **после** того как `pipeline-50k-major_metrics-annotation.ipynb` завершён и `ground_truth_pool.json` заполнен.

Метрики (pool-GT, graded 0/1/2):
- Recall@10
- MRR
- nDCG@10

В двух режимах: retrieve-only (bi-encoder) и retrieve+rerank (bi + cross-encoder).


In [1]:
# Поднимаемся к корню thesis/, чтобы относительные пути работали из подпапки
import os
from pathlib import Path
_p = Path.cwd()
while _p.name != 'thesis' and _p.parent != _p:
    _p = _p.parent
if _p.name == 'thesis':
    os.chdir(_p)
print('CWD:', Path.cwd())


CWD: c:\Users\Admin\Documents\диплом\thesis


In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import math
import time
from pathlib import Path
from collections import OrderedDict, defaultdict

import numpy as np
import pandas as pd
import torch
import lancedb
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets

import transformers
from transformers.modeling_utils import PreTrainedModel
transformers.PreTrainedModel = PreTrainedModel

from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {DEVICE}')


DEVICE: cpu


In [3]:
# ==================== КОНФИГУРАЦИЯ ====================

# Выбранный bi-encoder для финального пайплайна (лучший по Spearman на golden_eval)
BI_ENCODER_PATH   = 'models/bi-encoder-e5-finetuned'
BI_ENCODER_TABLE  = 'e5-base-fine-tuned-50k'
BI_DOC_PREFIX     = 'passage: '   # E5 требует префиксы
BI_QUERY_PREFIX   = 'query: '

CROSS_ENCODER_PATH = 'models/final/cross-encoder'

LANCEDB_PATH  = './lancedb_store'

# Ground truth pool (собирается один раз, размечается вручную)
GT_PAIRS_JSON       = 'ground_truth_pairs.json'          # исходные описания товаров
POOL_CANDIDATES_JSON = 'benchmark/pipeline/pool_candidates.json'  # сам pool (генерируется)
GT_POOL_JSON        = 'benchmark/pipeline/ground_truth_pool.json' # разметка (заполняется руками)

# Индексы описаний из ground_truth_pairs.json, которые НЕ используем
# (#6 — дубль #5 жиросжигатель; #14 — дубль #13 эзотерика)
DROP_QUERY_INDEXES = {6, 14}

# Сколько постов берём в pool от каждого режима
TOP_K_BI     = 20    # top-K retrieve-only
TOP_K_RERANK = 20    # top-K после cross-encoder (pool = union этих двух)

# Для cross-encoder reranking'а bi-encoder отдаёт больше кандидатов
TOP_K_BI_FOR_RERANK = 100

# Метрики считаются @10
K_METRIC = 10


## 1. Запросы

17 описаний товаров (из 19 исходных убраны дубли #6 «жиросжигатель» и #14 «скретч-открытки»).

In [4]:
# Загружаем описания товаров, исключаем дубли
with open(GT_PAIRS_JSON, encoding='utf-8') as f:
    all_pairs = json.load(f)

queries = []
for idx, pair in enumerate(all_pairs, 1):
    if idx in DROP_QUERY_INDEXES:
        continue
    queries.append({
        'query_idx':    idx,
        'imt_name':     pair.get('imt_name', ''),
        'description':  pair['description'],
    })

print(f'Запросов для пайплайна: {len(queries)} (из {len(all_pairs)}, убраны: {sorted(DROP_QUERY_INDEXES)})')
for q in queries:
    print(f"  {q['query_idx']:2d}. {q['imt_name']}")


Запросов для пайплайна: 17 (из 19, убраны: [6, 14])
   1. Затирка для плитки готовая - белая
   2. Самоклеящиеся панели для стен на кухню 60х30см пвх 15шт
   3. Развивашки 2-3-4 года/пиши стирай тетрадь/книги для малышей
   4. Детская мозаика (5 цветов, 40 элементов) "Кораблик"
   5. Жиросжигатель для похудения женщинам 60 капсул
   7. Накидка на сиденье DongFeng Fengshen Yixuan GS
   8. Кроссовер Monjaro
   9. Матрас надувной двуспальный 203х152см с подушками и насосом
  10. Гуд Найт Мягкое фито снотворное для сна
  11. Клавиатура игровая с подсветкой

  12. Видеокарта RTX 3050 6 ГБ RTL (RTX 3050 LP E 6G OC)
  13. Карты таро уэйта для начинающих с инструкцией обучающие
  15. Подушка для путешествий Travel Blue Tranquility Pillow (212)
  16. Чехол на чемодан L плотный на молнии с рисунком
  17. Спиннинг на щуку для рыбалки КATANA 2,1 м 5-25 г 15-40 г
  18. Кормушка для рыбалки Флэт - монтаж карповый фидерный
  19. Шляпа с декоративной цепочкой



## 2. Загрузка моделей

In [5]:
# Загружаем bi-encoder и cross-encoder
print(f'Загрузка bi-encoder: {BI_ENCODER_PATH}')
bi_encoder = SentenceTransformer(BI_ENCODER_PATH, device=DEVICE)
print(f'  dim={bi_encoder.get_sentence_embedding_dimension()}')

print(f'Загрузка cross-encoder: {CROSS_ENCODER_PATH}')
cross_encoder = CrossEncoder(CROSS_ENCODER_PATH, device=DEVICE)
print('  готово')

# LanceDB
db = lancedb.connect(LANCEDB_PATH)
assert BI_ENCODER_TABLE in db.table_names(), f'Нет таблицы {BI_ENCODER_TABLE}'
table = db.open_table(BI_ENCODER_TABLE)
print(f'Таблица {BI_ENCODER_TABLE}: {table.count_rows():,} постов')


You are trying to use a model that was created with Sentence Transformers version 5.3.0, but you're currently using version 5.2.3. This might cause unexpected behavior or errors. In that case, try to update to the latest version.


Загрузка bi-encoder: models/bi-encoder-e5-finetuned
  dim=768
Загрузка cross-encoder: models/final/cross-encoder


The tokenizer you are loading from 'models/final/cross-encoder' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


  готово
Таблица e5-base-fine-tuned-50k: 50,000 постов


In [6]:
def encode_query(q):
    q_in = (BI_QUERY_PREFIX + q) if BI_QUERY_PREFIX else q
    return bi_encoder.encode([q_in], normalize_embeddings=True)[0].tolist()


def retrieve(query, k):
    qvec = encode_query(query)
    return (table.search(qvec, query_type='vector')
                 .limit(k)
                 .select(['text', 'channel', 'category'])
                 .to_list())


def rerank(query, candidates):
    pairs = [[query, c['text']] for c in candidates]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    order = np.argsort(-np.asarray(scores))
    reranked = [candidates[i] for i in order]
    return reranked, [float(scores[i]) for i in order]


## Загрузка собранного pool


In [7]:
# Загружаем pool, собранный в annotation-ноутбуке
assert os.path.exists(POOL_CANDIDATES_JSON), (
    f'{POOL_CANDIDATES_JSON} не найден. Сначала прогони annotation-ноутбук '
    '(pipeline-50k-major_metrics-annotation.ipynb), чтобы собрать pool.'
)
with open(POOL_CANDIDATES_JSON, encoding='utf-8') as f:
    pool = json.load(f)
print(f'Pool загружен: {len(pool)} запросов, всего пар: {sum(len(p["candidates"]) for p in pool)}')


Pool загружен: 17 запросов, всего пар: 446


## 5. Метрики пайплайна

После того как pool размечен (или размечено большинство), считаем метрики в двух режимах.

In [8]:
# ============================================================
# РАСЧЁТ МЕТРИК на pool-GT
# ============================================================
# Требует, чтобы GT_POOL_JSON был заполнен (score 0/1/2 для всех/большинства пар).
# Считает Recall@10, MRR, nDCG@10 в двух режимах:
#   - retrieve-only   (только bi-encoder)
#   - retrieve+rerank (bi-encoder + cross-encoder)

assert os.path.exists(GT_POOL_JSON), f'Сначала разметь pool в ячейке выше ({GT_POOL_JSON} не найден)'
with open(GT_POOL_JSON, encoding='utf-8') as f:
    annotations = json.load(f)
print(f'Загружено разметок: {len(annotations)}')


def relevance_for(query_idx, post_text, pool):
    '''Возвращает score 0/1/2 для пары (query_idx, post_text).
    Если пары не было в pool или она не размечена — возвращает 0 (default).'''
    for p in pool:
        if p['query_idx'] != query_idx:
            continue
        for ci, c in enumerate(p['candidates']):
            if c['text'].strip() == post_text.strip():
                key = f"{query_idx}:{ci}"
                return annotations.get(key, 0)
    return 0


def recall_at_k(relevances_in_top_k, all_relevant_count):
    '''Доля найденных релевантных от всех размеченных релевантных у запроса.'''
    if all_relevant_count == 0:
        return None
    found = sum(1 for r in relevances_in_top_k if r >= 1)
    return found / all_relevant_count


def reciprocal_rank(relevances_in_top_k):
    '''1/ранг первого релевантного (score >= 1). 0 если не найден.'''
    for i, r in enumerate(relevances_in_top_k, 1):
        if r >= 1:
            return 1.0 / i
    return 0.0


def ndcg_at_k(relevances_in_top_k, all_relevances, k):
    '''Graded nDCG: gain = score (0/1/2).'''
    dcg = sum((2**rel - 1) / math.log2(i + 2) for i, rel in enumerate(relevances_in_top_k[:k]))
    ideal = sorted(all_relevances, reverse=True)[:k]
    idcg = sum((2**rel - 1) / math.log2(i + 2) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_run(run_results, pool, k=K_METRIC):
    '''run_results: list[{query_idx, top_k_texts}] — упорядоченный список постов в top-k.'''
    recalls, rrs, ndcgs = [], [], []
    for r in run_results:
        qi = r['query_idx']
        top_texts = r['top_k_texts']

        # размеченные релевантные этого запроса в pool (всё, что >= 1)
        all_rels = []
        for p in pool:
            if p['query_idx'] != qi:
                continue
            for ci, c in enumerate(p['candidates']):
                key = f"{qi}:{ci}"
                score = annotations.get(key, 0)
                if score >= 1:
                    all_rels.append(score)
        n_relevant = len(all_rels)
        if n_relevant == 0:
            # нет релевантных → метрики на этом запросе не определены, пропускаем
            continue

        # relevances в порядке top_k (из pool; если пост не в pool — score=0)
        relevances_in_top = [relevance_for(qi, t, pool) for t in top_texts]

        recalls.append(recall_at_k(relevances_in_top, n_relevant))
        rrs.append(reciprocal_rank(relevances_in_top))
        ndcgs.append(ndcg_at_k(relevances_in_top, all_rels, k))

    return {
        'Recall@10': float(np.mean(recalls)) if recalls else 0.0,
        'MRR':       float(np.mean(rrs))     if rrs     else 0.0,
        'nDCG@10':   float(np.mean(ndcgs))   if ndcgs   else 0.0,
        'n_queries': len(recalls),
    }


# Прогоняем пайплайн дважды на тех же запросах
retrieve_only_runs = []
rerank_runs = []

for q in queries:
    # retrieve-only: top-10
    ret = retrieve(q['description'], K_METRIC)
    retrieve_only_runs.append({
        'query_idx': q['query_idx'],
        'top_k_texts': [r['text'] for r in ret],
    })

    # retrieve+rerank: bi top-100 → cross-encoder → top-10
    cand100 = retrieve(q['description'], TOP_K_BI_FOR_RERANK)
    reranked, _ = rerank(q['description'], cand100)
    rerank_runs.append({
        'query_idx': q['query_idx'],
        'top_k_texts': [r['text'] for r in reranked[:K_METRIC]],
    })

m_retrieve = evaluate_run(retrieve_only_runs, pool)
m_rerank   = evaluate_run(rerank_runs,        pool)

df = pd.DataFrame([
    {'Режим': 'retrieve-only (bi-encoder)',         **{k: round(v, 4) for k, v in m_retrieve.items() if k != 'n_queries'}},
    {'Режим': 'retrieve+rerank (bi + cross)',       **{k: round(v, 4) for k, v in m_rerank.items()   if k != 'n_queries'}},
])
print(f'Метрики пайплайна на pool-GT ({m_retrieve["n_queries"]} запросов с хотя бы одним размеченным релевантным)')
display(df.style.background_gradient(cmap='YlGn', subset=['Recall@10', 'MRR', 'nDCG@10']))

print('\nДельта (retrieve+rerank − retrieve-only):')
for k in ['Recall@10', 'MRR', 'nDCG@10']:
    d = m_rerank[k] - m_retrieve[k]
    arrow = '↑' if d > 0 else ('↓' if d < 0 else '=')
    print(f'  Δ {k}: {d:+.4f}  {arrow}')


Загружено разметок: 446
Метрики пайплайна на pool-GT (17 запросов с хотя бы одним размеченным релевантным)


,Режим,Recall@10,MRR,nDCG@10
0,retrieve-only (bi-encoder),0.469600,0.746100,0.558500
1,retrieve+rerank (bi + cross),0.560600,0.854900,0.647800



Дельта (retrieve+rerank − retrieve-only):
  Δ Recall@10: +0.0909  ↑
  Δ MRR: +0.1088  ↑
  Δ nDCG@10: +0.0893  ↑
